In [2]:
import pandas as pd
import networkx as nx

# Load and Build Weighted Graph
df = pd.read_csv('data/airline.csv')
df = df[['Source airport', 'Destination airport']].dropna()

print(df.head())  # Check the first few rows to ensure correct loading

edges = df.groupby(['Source airport', 'Destination airport']).size().reset_index(name='weight')

# Gw is our weighted directed graph
Gw = nx.from_pandas_edgelist(edges, 'Source airport', 'Destination airport', edge_attr='weight', create_using=nx.DiGraph())

# Compute Centralities
print("\nComputing centralities...")

# Unweighted degree (total connections)
deg = nx.degree_centrality(Gw)

# Weighted Betweenness (Approximated with k=200)
# Note: NetworkX uses weight as "distance," so for "importance," we use 1/weight
for u, v, d in Gw.edges(data=True):
    d['inv_w'] = 1.0 / d['weight']
bet = nx.betweenness_centrality(Gw, k=200, normalized=True, weight='inv_w', seed=42)

# Weighted Eigenvector
eig = nx.eigenvector_centrality(Gw, max_iter=2000, tol=1e-6, weight="weight")

# Extract and Print Top 10
def print_top_10(centrality_dict, name):
    top_nodes = sorted(centrality_dict.items(), key=lambda x: x[1], reverse=True)[:10]
    print(f"\nTop 10 {name}:")
    for node, val in top_nodes:
        print(f"{node}: {val:.4f}")

print_top_10(deg, "Degree Centrality")
print_top_10(bet, "Betweenness Centrality")
print_top_10(eig, "Eigenvector Centrality")

  Source airport Destination airport
0            AER                 KZN
1            ASF                 KZN
2            ASF                 MRV
3            CEK                 KZN
4            CEK                 OVB

Computing centralities...

Top 10 Degree Centrality:
FRA: 0.1393
CDG: 0.1373
AMS: 0.1352
IST: 0.1335
ATL: 0.1265
PEK: 0.1203
ORD: 0.1195
MUC: 0.1110
DME: 0.1104
DFW: 0.1086

Top 10 Betweenness Centrality:
LHR: 0.2481
LAX: 0.1723
ATL: 0.1711
JFK: 0.1653
SIN: 0.1052
ORD: 0.0893
SEA: 0.0845
MIA: 0.0826
ANC: 0.0772
GRU: 0.0696

Top 10 Eigenvector Centrality:
ATL: 0.2703
LHR: 0.2087
ORD: 0.2006
JFK: 0.1941
LAX: 0.1885
CDG: 0.1577
DFW: 0.1453
FRA: 0.1430
SFO: 0.1262
YYZ: 0.1252
